# CESM2 NAO

In [2]:
import os
import sys
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr

In [3]:
sys.path.insert(0, os.path.realpath('../libs/'))
import graph_utils as gu
#import verif_utils as vu

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
from eofs.xarray import Eof

In [9]:
list_nao = []

for ilead in range(10):
    list_ds = []

    for year in range(1958, 2020):
        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
        ds = xr.open_zarr(fn)[['PSL']]
        ds = ds.mean('member')

        year_pick = year + ilead + 1
        ds = ds.sel(time=slice(f'{year_pick}-01-01', f'{year_pick}-12-31'))
        list_ds.append(ds)

    ds_all = xr.concat(list_ds, dim='time').sortby('time')

    # SLP in hPa and convert daily -> monthly
    slp = ds_all['PSL'] / 100.0
    slp = slp.resample(time="MS").mean()

    # convert lon to [-180, 180)
    slp = slp.assign_coords(
        lon=((slp.lon + 180) % 360) - 180
    ).sortby("lon")

    # robust lat slicing
    if slp.lat[0] < slp.lat[-1]:
        slp_na = slp.sel(lat=slice(20, 80), lon=slice(-90, 40))
    else:
        slp_na = slp.sel(lat=slice(80, 20), lon=slice(-90, 40))

    # monthly anomaly
    slp_mon_clim = slp_na.groupby("time.month").mean("time")
    anom = slp_na.groupby("time.month") - slp_mon_clim
    anom = anom.transpose("time", "lat", "lon")
    
    # weights
    w_lat = np.sqrt(np.cos(np.deg2rad(anom["lat"])))
    weights_2d = xr.DataArray(
        np.broadcast_to(w_lat.values[:, None], (anom.sizes["lat"], anom.sizes["lon"])),
        coords={"lat": anom["lat"], "lon": anom["lon"]},
        dims=("lat", "lon"),
    )
    
    # EOF
    solver = Eof(anom, weights=weights_2d)
    eof1 = solver.eofs(neofs=1)[0]
    pc1 = solver.pcs(npcs=1, pcscaling=1)[:, 0]

    # sign convention using simple monthly box NAO
    if slp.lat[0] < slp.lat[-1]:
        south = slp.sel(lat=slice(35, 40), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(60, 70), lon=slice(-30, -10)).mean(("lat", "lon"))
    else:
        south = slp.sel(lat=slice(40, 35), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(70, 60), lon=slice(-30, -10)).mean(("lat", "lon"))

    nao_box = south - north
    nao_box = nao_box.groupby("time.month") - nao_box.groupby("time.month").mean("time")

    corr = xr.corr(pc1, nao_box)
    if float(corr) < 0:
        eof1 = -eof1
        pc1 = -pc1

    # pc1 is already monthly
    nao_monthly = (pc1 - pc1.mean("time")) / pc1.std("time")
    nao_monthly.name = "nao_monthly_eof"

    # annual
    nao_ann = nao_monthly.groupby("time.year").mean("time")
    nao_ann.name = "nao_ann_eof"

    # DJF
    years = np.asarray(np.unique(nao_monthly["time.year"].values))
    first_year = int(years.min())
    
    pieces = []
    jf_first = nao_monthly.where(
        (nao_monthly["time.year"] == first_year) &
        (nao_monthly["time.month"].isin([1, 2])),
        drop=True
    )

    if jf_first.sizes.get("time", 0) == 2:
        jf_first_mean = jf_first.mean("time")
        jf_first_mean = jf_first_mean.expand_dims(year=[first_year])
        pieces.append(jf_first_mean)
        
    djf_raw = (
        nao_monthly.where(nao_monthly["time.month"].isin([12, 1, 2]), drop=True)
                   .resample(time="QS-DEC")
    )

    djf = djf_raw.mean()
    djf_count = djf_raw.count()

    djf = djf.where(djf_count == 3, drop=True)
    djf = djf.where(djf["time.month"] == 12, drop=True)
    djf = djf.assign_coords(year=djf["time.year"] + 1)
    djf = djf.swap_dims({"time": "year"}).drop_vars("time")
    djf = djf.dropna("year")

    # avoid duplicate first year if it somehow exists
    djf = djf.where(djf["year"] != first_year, drop=True)
    pieces.append(djf)

    nao_djf = xr.concat(pieces, dim="year").sortby("year")
    nao_djf.name = "nao_djf_eof"
    

    # other seasons
    nao_mam = nao_monthly.where(nao_monthly["time.month"].isin([3, 4, 5]), drop=True).groupby("time.year").mean("time")
    nao_jja = nao_monthly.where(nao_monthly["time.month"].isin([6, 7, 8]), drop=True).groupby("time.year").mean("time")
    nao_son = nao_monthly.where(nao_monthly["time.month"].isin([9, 10, 11]), drop=True).groupby("time.year").mean("time")

    nao_mam.name = "nao_mam_eof"
    nao_jja.name = "nao_jja_eof"
    nao_son.name = "nao_son_eof"

    ds_nao_yearly = xr.Dataset({
        "nao_ann_eof": nao_ann,
        "nao_djf_eof": nao_djf,
        "nao_mam_eof": nao_mam,
        "nao_jja_eof": nao_jja,
        "nao_son_eof": nao_son,
    })
    
    ds_nao_yearly = ds_nao_yearly.rename({'year': 'init_year'})
    ds_nao_yearly = ds_nao_yearly.assign_coords({'init_year': np.arange(1958, 2020)})
    
    list_nao.append(ds_nao_yearly)
    print(f'lead year {ilead + 1} done')
    
ds_nao_all = xr.concat(
    list_nao,
    dim=xr.IndexVariable("lead_year", np.arange(10))
)

lead year 1 done
lead year 2 done
lead year 3 done
lead year 4 done
lead year 5 done
lead year 6 done
lead year 7 done
lead year 8 done
lead year 9 done
lead year 10 done


In [11]:
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO.zarr'

ds_nao_all = ds_nao_all.chunk({'init_year': 62, 'lead_year': 10})
# ds_nao_all.to_zarr(save_name)